In [ ]:
import pandas as pd
import numpy as np
import torch
from functools import reduce
import warnings

# 設定環境
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. 讀取與合併資料 (Load & Merge)
# ==========================================
print("步驟 1/3: 正在讀取並合併檔案...")

base_path = "C:\\Users\\user\\Desktop\\HW-NCHU\\meeting\\GPR\\data\\"

# 定義檔案路徑
file_paths = {
    "PM25":   base_path + "PM2.5_FPCA_2025.csv", # Input Feature 1 & Train Target
    "WIND_U": base_path + "WIND_U_FPCA_2025.csv",     # Input Feature 2
    "WIND_V": base_path + "WIND_V_FPCA_2025.csv",     # Input Feature 3
    "RH":     base_path + "RH_FPCA_2025.csv",    # Input Feature 4
    "TEMP":   base_path + "AMB_TEMP_FPCA_2025.csv"     # Input Feature 5
}

# --- A. 處理 5 個 FPCA 特徵 ---
dfs_features = []
for feature_name, path in file_paths.items():
    df = pd.read_csv(path)
    # 移除多餘欄位，只留時間跟站點數據
    cols = [c for c in df.columns if c not in ['date', 'Time', 'year', 'SubjectID']]
    df = df[cols]
    df['PublishTime'] = pd.to_datetime(df['PublishTime'])
    
    # 轉成長表格 (Long Format): [PublishTime, Station, Feature_Value]
    df_melt = df.melt(id_vars=['PublishTime'], var_name='Station', value_name=feature_name)
    dfs_features.append(df_melt)

# 合併這 5 個表格 (Inner Join)
df_merged_features = reduce(lambda left, right: pd.merge(left, right, on=['PublishTime', 'Station'], how='inner'), dfs_features)

# --- B. 處理 Raw Data (用於 Test 驗證) ---
raw_df = pd.read_csv(base_path + "PM2.5.csv")
raw_df['PublishTime'] = pd.to_datetime(raw_df['PublishTime'])
# 轉長表格
raw_melted = raw_df.melt(id_vars=['PublishTime'], var_name='Station', value_name='PM25_Raw')

# --- C. 最終合併 ---
# 把 Raw Data 合併進去 (Left Join)
df_all = pd.merge(df_merged_features, raw_melted, on=['PublishTime', 'Station'], how='left')

# 確保欄位順序固定: [PM25, U, V, RH, TEMP, PM25_Raw]
# Index 對照: 0=PM25, 1=U, 2=V, 3=RH, 4=TEMP, 5=Raw
cols_order = ['PublishTime', 'Station', 'PM25', 'WIND_U', 'WIND_V', 'RH', 'TEMP', 'PM25_Raw']
df_all = df_all[cols_order]

# 排序
df_all = df_all.sort_values(by=['Station', 'PublishTime'])

print("合併完成！資料預覽：")
print(df_all.head())

# ==========================================
# 2. 滑動視窗切分 (Sliding Window) - 已修改為不重疊
# ==========================================
print("\n步驟 2/3: 正在切分時間視窗 (每 24 小時跳一次)...")

# 參數設定
INPUT_HOURS = 24
OUTPUT_HOURS = 72
TOTAL_WINDOW = INPUT_HOURS + OUTPUT_HOURS # 48
STEP_SIZE = 24  

SPLIT_DATE = pd.Timestamp('2025-01-01')
TRAIN_START = pd.Timestamp('2018-01-01')
TEST_END = pd.Timestamp('2025-11-30')

# 準備存放結果的字典
station_datasets = {}

# 取得所有站點清單
stations = df_all['Station'].unique()

for station in stations:
    # 取出單一站點資料
    df_station = df_all[df_all['Station'] == station].copy()
    df_station = df_station.set_index('PublishTime').sort_index()
    
    # 強制補齊時間軸
    # 加上內層的中括號
    df_station = df_station[['PM25', 'WIND_U', 'WIND_V', 'RH', 'TEMP', 'PM25_Raw']].asfreq('H')
    
    # 轉成 Numpy Array 加速
    data_values = df_station.values  # Shape: (T, 6)
    times = df_station.index
    
    # 暫存 list
    train_X_list, train_Y_list = [], []
    test_X_list, test_Y_raw_list = [], []
    
    num_samples = len(data_values) - TOTAL_WINDOW + 1
    
    if num_samples <= 0:
        continue

    # ============================================================
    # 【這裡改了】加上 step (第三個參數)
    # range(start, stop, step)
    # i 會變成: 0, 24, 48, 72... 
    # 這樣視窗就不會重疊：
    # i=0:  0~23(Input) -> 24~47(Output)
    # i=24: 24~47(Input) -> 48~71(Output)
    # ============================================================
    for i in range(0, num_samples, STEP_SIZE):
        
        window = data_values[i : i + TOTAL_WINDOW]
        current_time = times[i]
        
        # --- 1. 處理 Input (前 24 小時) ---
        x_window = window[:INPUT_HOURS, 0:5]
        
        # 如果 Input 有任何 NaN，跳過 (這一天資料不全，不預測)
        if np.isnan(x_window).any():
            continue
            
        # --- 2. 處理 Output (後 24 小時) ---
        if TRAIN_START <= current_time < SPLIT_DATE:
            # === Training Set ===
            y_window_smooth = window[INPUT_HOURS:, 0] # 這裡會自動切出 72 小時
            
            if np.isnan(y_window_smooth).any():
                continue
            
            train_X_list.append(x_window)
            train_Y_list.append(y_window_smooth)
            
        elif SPLIT_DATE <= current_time <= (TEST_END - pd.Timedelta(hours=TOTAL_WINDOW)):
            # === Testing Set ===
            y_window_raw = window[INPUT_HOURS:, 5] # 這裡會自動切出 72 小時
            
            if np.isnan(y_window_raw).all():
                continue
                
            test_X_list.append(x_window)
            test_Y_raw_list.append(y_window_raw)
    
    # 存入字典
    if len(train_X_list) > 0 and len(test_X_list) > 0:
        station_datasets[station] = {
            'train_x': torch.tensor(np.array(train_X_list), dtype=torch.float32).to(device),
            'train_y': torch.tensor(np.array(train_Y_list), dtype=torch.float32).to(device),
            'test_x':  torch.tensor(np.array(test_X_list), dtype=torch.float32).to(device),
            'test_y_raw': np.array(test_Y_raw_list) 
        }

print(f"切分完成！共有 {len(station_datasets)} 個站點有有效資料。")


# ==========================================
# 3. 檢查結果 (Verification)
# ==========================================
print("\n步驟 3/3: 檢查所有站點的資料形狀...")

if len(station_datasets) > 0:
    print(f"{'='*60}")
    # 使用 items() 同時取得 站點名稱(name) 和 資料(data)
    for station_name, data in station_datasets.items():
        print(f"站點: {station_name}")
        print(f"  Train Input Shape: {data['train_x'].shape}")
        print(f"  Train Label Shape: {data['train_y'].shape}")
        print(f"  Test  Input Shape: {data['test_x'].shape}")
        print(f"  Test  Label Shape: {data['test_y_raw'].shape} (含 NaN)")
        print(f"{'-'*30}") # 分隔線，讓閱讀更清楚
        
    print(f"\n檢查完畢！共 {len(station_datasets)} 個站點。")
else:
    print("警告：沒有產生任何有效資料。")

# FEDONet

use fourier embedding deeponet and let the parameter learning from data because is better than origin random normal parameter


In [ ]:
"""
DeepONet — 全站聯合訓練 + 經緯度放 Branch + Random Fourier Trunk
================================================================
相比原本的版本，唯一改動：
  Trunk 時間編碼：5 維物理週期 → 2m+1 維 Random Fourier Features
  頻率從 N(0, sigma^2) 隨機抽樣，固定不訓練
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gc
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =============================================================================
# 0. 讀取站點座標
# =============================================================================
station_info = pd.read_csv(r"C:\Users\user\Desktop\HW-NCHU\meeting\GPR\data\station .csv",
                           encoding='utf-8-sig')

station_coords = {}
for _, row in station_info.iterrows():
    station_coords[row['SITE_NAME']] = (row['lat'], row['lon'])

print(f"載入 {len(station_coords)} 個站點座標")
for name in list(station_coords.keys())[:5]:
    print(f"  {name}: lat={station_coords[name][0]}, lon={station_coords[name][1]}")


# =============================================================================
# 1. 合併全站資料 + 附加經緯度
# =============================================================================
print("\n合併全站資料...")

all_train_x = []
all_train_y = []
all_test_x  = []
all_test_y_raw = []
all_test_station = []

NUM_VARS = 5
skipped = []

for station_name, dataset in station_datasets.items():
    if station_name not in station_coords:
        skipped.append(station_name)
        continue

    lat, lon = station_coords[station_name]

    train_x = dataset['train_x'][:, :, :NUM_VARS].cpu().numpy()
    train_y = dataset['train_y'].cpu().numpy()
    test_x  = dataset['test_x'][:, :, :NUM_VARS].cpu().numpy()
    test_y_raw = dataset['test_y_raw']

    N_train = train_x.shape[0]
    N_test  = test_x.shape[0]

    train_x_flat = train_x.reshape(N_train, -1)
    test_x_flat  = test_x.reshape(N_test, -1)

    train_coords = np.full((N_train, 2), [lat, lon])
    test_coords  = np.full((N_test, 2),  [lat, lon])

    train_x_with_coords = np.concatenate([train_x_flat, train_coords], axis=1)
    test_x_with_coords  = np.concatenate([test_x_flat,  test_coords],  axis=1)

    all_train_x.append(train_x_with_coords)
    all_train_y.append(train_y)
    all_test_x.append(test_x_with_coords)
    all_test_y_raw.append(test_y_raw)
    all_test_station.extend([station_name] * N_test)

if skipped:
    print(f"警告：以下站點在 station_.csv 中找不到座標，已跳過: {skipped}")

all_train_x    = np.concatenate(all_train_x, axis=0)
all_train_y    = np.concatenate(all_train_y, axis=0)
all_test_x     = np.concatenate(all_test_x, axis=0)
all_test_y_raw = np.concatenate(all_test_y_raw, axis=0)
all_test_station = np.array(all_test_station)

print(f"合併完成！")
print(f"  Train: {all_train_x.shape[0]} 筆 × {all_train_x.shape[1]} 維")
print(f"  Test:  {all_test_x.shape[0]} 筆 × {all_test_x.shape[1]} 維")
print(f"  涵蓋站點數: {len(set(all_test_station))}")


# =============================================================================
# 2. 轉 Tensor
# =============================================================================
train_x_t = torch.tensor(all_train_x, dtype=torch.float32).to(device)
train_y_t = torch.tensor(all_train_y, dtype=torch.float32).to(device)
test_x_t  = torch.tensor(all_test_x,  dtype=torch.float32).to(device)


# =============================================================================
# 3. 模型：Random Fourier Features Trunk
# =============================================================================
# Fourier 超參數
M = 32      

class DeepONet_Fourier(nn.Module):
    def __init__(self, input_dim=122, p=128, m=64):
        super().__init__()

        self.freqs = nn.Parameter(torch.empty(m).uniform_(0.0, 2.0))

        trunk_in_dim = 2 * m + 1  # sin + cos + t_norm

        # Branch
        self.branch = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, p),
        )

        # Trunk
        self.trunk = nn.Sequential(
            nn.Linear(trunk_in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, p),
        )

        self.bias = nn.Parameter(torch.zeros(1))

    def _encode_time(self):
        """建構 Random Fourier 時間編碼"""
        t = torch.arange(1, 73, dtype=torch.float32, device=self.freqs.device)  # (72,)
        t_norm = (t / 72).unsqueeze(-1)                                          # (72, 1)
        angles = 2 * np.pi * t.unsqueeze(-1) * self.freqs                        # (72, m)
        encoding = torch.cat([t_norm, torch.sin(angles), torch.cos(angles)], dim=-1)  # (72, 2m+1)
        return encoding

    def forward(self, branch_input, trunk_input=None):
        # trunk_input 參數不再使用，改為內部建構
        b = self.branch(branch_input)
        t = self.trunk(self._encode_time())
        output = torch.matmul(b, t.T) + self.bias
        return output


# =============================================================================
# 4. 訓練
# =============================================================================
P = 128
LR = 0.001
TRAINING_ITER = 2000
WEIGHT_DECAY = 1e-2

print(f"\n{'='*10} DeepONet 全站聯合 + 經緯度 + Random Fourier Trunk (m={M}, σ={SIGMA}) {'='*10}")
print(f"Train size: {train_x_t.shape[0]} | Test size: {test_x_t.shape[0]}")

model = DeepONet_Fourier(input_dim=122, p=P, m=M).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAINING_ITER, eta_min=1e-5
)
criterion = nn.MSELoss()

model.train()
for epoch in range(TRAINING_ITER):
    optimizer.zero_grad()
    preds = model(train_x_t)                     # 不再傳 trunk_input
    loss = criterion(preds, train_y_t)
    loss.backward()
    optimizer.step()
    scheduler.step()

    if (epoch + 1) % 200 == 0:
        print(f"  Epoch {epoch+1}/{TRAINING_ITER} | Loss: {loss.item():.6f}")


# =============================================================================
# 5. 評估
# =============================================================================
model.eval()
with torch.no_grad():
    y_pred_all = model(test_x_t).cpu().numpy()

results_list = []
unique_stations = sorted(set(all_test_station))

for station_name in unique_stations:
    mask_station = (all_test_station == station_name)
    y_true = all_test_y_raw[mask_station]
    y_pred = y_pred_all[mask_station]

    mask_valid = ~np.isnan(y_true)
    if mask_valid.sum() > 0:
        mae  = mean_absolute_error(y_true[mask_valid], y_pred[mask_valid])
        rmse = np.sqrt(mean_squared_error(y_true[mask_valid], y_pred[mask_valid]))
        results_list.append({'Station': station_name, 'MAE': mae, 'RMSE': rmse})

results_df = pd.DataFrame(results_list)

print(f"\n{'='*20} 逐站結果 {'='*20}")
for _, row in results_df.iterrows():
    print(f"  {row['Station']:<6} | MAE: {row['MAE']:.4f} | RMSE: {row['RMSE']:.4f}")

print(f"\n{'='*20} 最終結果 (Random Fourier Trunk) {'='*20}")
print(f"平均 MAE : {results_df['MAE'].mean():.4f}")
print(f"平均 RMSE: {results_df['RMSE'].mean():.4f}")
print(f"站點數: {len(results_df)}")

del model, optimizer, scheduler, train_x_t, train_y_t, test_x_t, y_pred_all
gc.collect()
torch.cuda.empty_cache()

MAE=5.3628  | RMSE=7.3967  

# POD-DeepONet

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gc
from sklearn.metrics import mean_absolute_error, mean_squared_error


# =============================================================================
# 0. 讀取站點座標（不動）
# =============================================================================
station_info = pd.read_csv(r"C:\Users\user\Desktop\HW-NCHU\meeting\GPR\data\station .csv",
                           encoding='utf-8-sig')

station_coords = {}
for _, row in station_info.iterrows():
    station_coords[row['SITE_NAME']] = (row['lat'], row['lon'])

print(f"載入 {len(station_coords)} 個站點座標")


# =============================================================================
# 1. 合併全站資料 + 附加經緯度（不動）
# =============================================================================
print("\n合併全站資料...")

all_train_x, all_train_y, all_test_x, all_test_y_raw, all_test_station = [], [], [], [], []
NUM_VARS = 5
skipped = []

for station_name, dataset in station_datasets.items():
    if station_name not in station_coords:
        skipped.append(station_name)
        continue

    lat, lon = station_coords[station_name]
    train_x = dataset['train_x'][:, :, :NUM_VARS].cpu().numpy()
    train_y = dataset['train_y'].cpu().numpy()
    test_x  = dataset['test_x'][:, :, :NUM_VARS].cpu().numpy()
    test_y_raw = dataset['test_y_raw']

    N_train, N_test = train_x.shape[0], test_x.shape[0]

    train_x_flat = train_x.reshape(N_train, -1)
    test_x_flat  = test_x.reshape(N_test, -1)

    train_coords = np.full((N_train, 2), [lat, lon])
    test_coords  = np.full((N_test, 2),  [lat, lon])

    train_x_with_coords = np.concatenate([train_x_flat, train_coords], axis=1)
    test_x_with_coords  = np.concatenate([test_x_flat,  test_coords],  axis=1)

    all_train_x.append(train_x_with_coords)
    all_train_y.append(train_y)
    all_test_x.append(test_x_with_coords)
    all_test_y_raw.append(test_y_raw)
    all_test_station.extend([station_name] * N_test)

all_train_x    = np.concatenate(all_train_x, axis=0)
all_train_y    = np.concatenate(all_train_y, axis=0)
all_test_x     = np.concatenate(all_test_x, axis=0)
all_test_y_raw = np.concatenate(all_test_y_raw, axis=0)
all_test_station = np.array(all_test_station)

# =============================================================================
# 2. 轉 Tensor（不動）
# =============================================================================
train_x_t = torch.tensor(all_train_x, dtype=torch.float32).to(device)
train_y_t = torch.tensor(all_train_y, dtype=torch.float32).to(device)
test_x_t  = torch.tensor(all_test_x,  dtype=torch.float32).to(device)


# =============================================================================
# 3. POD 模組
# =============================================================================
class POD_1D:
    def __init__(self, n_components=20):
        self.p = n_components
        self.basis = None    # shape: (72, p)
        self.y_mean = None   # shape: (72,)

    def fit(self, Y):
        self.y_mean = Y.mean(dim=0)
        Y_centered = Y - self.y_mean
        U, S, Vt = torch.linalg.svd(Y_centered, full_matrices=False)

        max_p = Vt.shape[0]
        if self.p > max_p:
            self.p = max_p

        self.basis = Vt[:self.p, :].T  # (72, p)

        explained = S**2 / (S**2).sum()
        cumulative = explained.cumsum(dim=0)
        print(f"  POD: top {self.p} basis 解釋了 {cumulative[self.p-1]*100:.2f}% 的變異")


# =============================================================================
# 4. POD-DeepONet：只有 Branch，沒有 Trunk
# =============================================================================
class POD_DeepONet(nn.Module):
    def __init__(self, input_dim, p, basis, y_mean):
        """
        basis:  (72, p)  固定的 POD 基底，從 SVD 取得
        y_mean: (72,)    訓練資料的平均
        """
        super().__init__()

        # 把 basis 和 y_mean 註冊成 buffer（不參與訓練、會跟著 .to(device) 移動）
        self.register_buffer('basis', basis)
        self.register_buffer('y_mean', y_mean)

        # Branch：輸入 u → p 個 POD 係數
        self.branch = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, p),
        )

    def forward(self, branch_input):
        coeffs = self.branch(branch_input)              # (N, p) 預測的 POD 係數
        output = torch.matmul(coeffs, self.basis.T)     # (N, 72) 重構
        output = output + self.y_mean                   # 加回平均
        return output


# =============================================================================
# 5. 訓練
# =============================================================================
P = 32
LR = 1e-3
TRAINING_ITER = 1500
WEIGHT_DECAY = 1e-2

print(f"\n{'='*10} POD-DeepONet (Lu et al. 2022) {'='*10}")

# 先做 POD
pod = POD_1D(n_components=P)
pod.fit(train_y_t)

# 建模型，把 POD 結果灌進去
model = POD_DeepONet(
    input_dim=train_x_t.shape[1],   # 122
    p=pod.p,
    basis=pod.basis,
    y_mean=pod.y_mean,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAINING_ITER, eta_min=1e-5
)
criterion = nn.MSELoss()

model.train()
for epoch in range(TRAINING_ITER):
    optimizer.zero_grad()
    preds = model(train_x_t)
    loss = criterion(preds, train_y_t)
    loss.backward()
    optimizer.step()
    scheduler.step()

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1}/{TRAINING_ITER} | Loss: {loss.item():.6f}")


# =============================================================================
# 6. 評估（不變）
# =============================================================================
model.eval()
with torch.no_grad():
    y_pred_all = model(test_x_t).cpu().numpy()

results_list = []
unique_stations = sorted(set(all_test_station))

for station_name in unique_stations:
    mask_station = (all_test_station == station_name)
    y_true = all_test_y_raw[mask_station]
    y_pred = y_pred_all[mask_station]

    mask_valid = ~np.isnan(y_true)
    if mask_valid.sum() > 0:
        mae  = mean_absolute_error(y_true[mask_valid], y_pred[mask_valid])
        rmse = np.sqrt(mean_squared_error(y_true[mask_valid], y_pred[mask_valid]))
        results_list.append({'Station': station_name, 'MAE': mae, 'RMSE': rmse})

results_df = pd.DataFrame(results_list)
print(f"\n{'='*20} 最終結果 (POD-DeepONet) {'='*20}")
print(f"平均 MAE : {results_df['MAE'].mean():.4f}")
print(f"平均 RMSE: {results_df['RMSE'].mean():.4f}")

#del model, optimizer, scheduler, y_pred_all
#gc.collect()
#torch.cuda.empty_cache()

MAE=5.5.69  | RMSE=7.4591  